# TAREA 1 — Simulación avanzada de inventarios de leche

**Autor:** Edward Calderon  
**Curso:** Simulación Avanzada  
**Entrega:** 10 de marzo de 2026

---

Este notebook presenta un análisis reproducible y comparativo en **R** y **Python** para resolver el problema de inventario descrito en la Tarea 2. El objetivo es encontrar una política de abastecimiento que minimice el costo total (inventario + suministro) y garantice al menos **80% de días sin faltantes**.

## 1️⃣ Introducción

El sistema opera 7 días por semana y enfrenta una demanda diaria aleatoria de leche en caja. La decisión de abastecimiento es estratégica porque impacta dos frentes:

1. **Nivel de servicio:** proporción de días sin faltante (meta ≥ 80%).
2. **Costo total:** suma de costo de inventario diario y costo logístico de cada despacho.

Modelamos el problema por **simulación Monte Carlo** durante 100 semanas y extendemos el análisis con una búsqueda de políticas para encontrar soluciones de menor costo bajo restricción de servicio.

## 2️⃣ Enunciado formal (Edward Calderon, Simulación Avanzada)

- Distribución empírica de demanda diaria a partir de 364 observaciones (52 semanas × 7 días).
- Contrato base: entrega semanal de 30 cajas los lunes.
- Horizonte de simulación: 100 semanas.
- Inventario inicial: 0 cajas (lunes en la mañana).
- Faltantes no se acumulan entre días.
- Costos:
  - Inventario: **$800 por caja por día**.
  - Suministro por entrega: **$15,000 fijos + $100 por caja entregada**.
- Política flexible permitida: entregas lunes y/o miércoles, con cantidades a decidir.
- Restricción: al menos **80% de días sin faltantes**.
- Objetivo: minimizar costo total esperado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
np.random.seed(2026)

# Distribución empírica (ventas en cajas, número de días)
demand_freq = pd.DataFrame({
    "demanda": [0,1,2,3,4,5,6,7,8,9,10],
    "frecuencia": [1,5,12,19,27,59,74,59,49,37,22]
})

demand_freq["probabilidad"] = demand_freq["frecuencia"] / demand_freq["frecuencia"].sum()
demand_freq["acumulada"] = demand_freq["probabilidad"].cumsum()

demand_freq

## 3️⃣ Distribución de demanda

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=demand_freq, x="demanda", y="probabilidad", color="#1f77b4", ax=ax[0])
ax[0].set_title("Distribución empírica de demanda diaria")
ax[0].set_xlabel("Demanda (cajas)")
ax[0].set_ylabel("Probabilidad")

sns.lineplot(data=demand_freq, x="demanda", y="acumulada", marker="o", color="#d62728", ax=ax[1])
ax[1].set_title("Función de distribución acumulada")
ax[1].set_xlabel("Demanda (cajas)")
ax[1].set_ylabel("Probabilidad acumulada")
ax[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Muestreo rápido desde la distribución empírica
values = demand_freq["demanda"].to_numpy()
probs = demand_freq["probabilidad"].to_numpy()

def sample_demands(n_days, rng):
    return rng.choice(values, size=n_days, p=probs)

## 4️⃣ Simulación en R

> Código equivalente en R (puede ejecutarse en Colab con `%%R` + `rpy2` o en RStudio).

```r
library(dplyr)

valores <- 0:10
freq <- c(1,5,12,19,27,59,74,59,49,37,22)
prob <- freq / sum(freq)

simular_politica <- function(weeks=100, q_lunes=30, q_miercoles=0,
                             costo_inv=800, costo_fijo=15000, costo_var=100,
                             semilla=2026) {
  set.seed(semilla)
  dias <- weeks * 7
  inventario <- 0
  sin_faltante <- 0
  costo_inv_total <- 0
  costo_sum_total <- 0

  for (d in 1:dias) {
    dia_semana <- ((d - 1) %% 7) + 1

    if (dia_semana == 1 && q_lunes > 0) {
      inventario <- inventario + q_lunes
      costo_sum_total <- costo_sum_total + costo_fijo + costo_var * q_lunes
    }
    if (dia_semana == 3 && q_miercoles > 0) {
      inventario <- inventario + q_miercoles
      costo_sum_total <- costo_sum_total + costo_fijo + costo_var * q_miercoles
    }

    demanda <- sample(valores, size=1, prob=prob)
    venta <- min(inventario, demanda)

    if (demanda <= inventario) sin_faltante <- sin_faltante + 1

    inventario <- inventario - venta
    costo_inv_total <- costo_inv_total + inventario * costo_inv
  }

  tibble(
    q_lunes=q_lunes,
    q_miercoles=q_miercoles,
    servicio=sin_faltante/dias,
    costo_total=costo_inv_total + costo_sum_total,
    costo_inventario=costo_inv_total,
    costo_suministro=costo_sum_total
  )
}

simular_politica(weeks=100, q_lunes=30, q_miercoles=0)
```

## 5️⃣ Simulación en Python

In [ ]:
from dataclasses import dataclass

@dataclass
class SimResult:
    q_lunes: int
    q_miercoles: int
    servicio: float
    costo_total: float
    costo_inventario: float
    costo_suministro: float


def simulate_policy(
    weeks=100,
    q_lunes=30,
    q_miercoles=0,
    inv_cost=800,
    fixed_order_cost=15000,
    variable_unit_cost=100,
    seed=2026,
):
    rng = np.random.default_rng(seed)
    days = weeks * 7
    inventory = 0

    no_stockout_days = 0
    inv_cost_total = 0.0
    supply_cost_total = 0.0

    for d in range(days):
        dow = d % 7

        if dow == 0 and q_lunes > 0:
            inventory += q_lunes
            supply_cost_total += fixed_order_cost + variable_unit_cost * q_lunes

        if dow == 2 and q_miercoles > 0:
            inventory += q_miercoles
            supply_cost_total += fixed_order_cost + variable_unit_cost * q_miercoles

        demand = rng.choice(values, p=probs)
        sold = min(inventory, demand)

        if demand <= inventory:
            no_stockout_days += 1

        inventory -= sold
        inv_cost_total += inventory * inv_cost

    return SimResult(
        q_lunes=q_lunes,
        q_miercoles=q_miercoles,
        servicio=no_stockout_days / days,
        costo_total=inv_cost_total + supply_cost_total,
        costo_inventario=inv_cost_total,
        costo_suministro=supply_cost_total,
    )

base = simulate_policy(weeks=100, q_lunes=30, q_miercoles=0)
pd.DataFrame([base.__dict__])

In [ ]:
# Intervalos de confianza por replicación independiente

def replicate_policy(n_rep=1000, **kwargs):
    rows = []
    base_seed = kwargs.pop("base_seed", 10000)
    for i in range(n_rep):
        r = simulate_policy(seed=base_seed + i, **kwargs)
        rows.append(r.__dict__)
    return pd.DataFrame(rows)

base_reps = replicate_policy(n_rep=1000, weeks=100, q_lunes=30, q_miercoles=0)
summary = base_reps[["servicio", "costo_total", "costo_inventario", "costo_suministro"]].agg(["mean", "std"])
summary

In [ ]:
def ci95(series):
    mean = series.mean()
    se = series.std(ddof=1) / np.sqrt(len(series))
    z = 1.96
    return pd.Series({"mean": mean, "ci_low": mean - z*se, "ci_high": mean + z*se})

ci_table = pd.DataFrame({
    "servicio": ci95(base_reps["servicio"]),
    "costo_total": ci95(base_reps["costo_total"]),
    "costo_inventario": ci95(base_reps["costo_inventario"]),
    "costo_suministro": ci95(base_reps["costo_suministro"]),
}).T

ci_table

## 6️⃣ Comparación R vs Python

Las dos implementaciones modelan exactamente la misma lógica:

- Entregas al inicio del día (lunes y opcional miércoles).
- Venta limitada por inventario disponible.
- Faltantes no acumulados.
- Costo de inventario calculado al final de cada día.
- Costo de suministro por cada despacho realizado.

La comparación esperada muestra resultados estadísticamente equivalentes (diferencias menores por variabilidad de semilla y generador aleatorio específico de cada lenguaje).

## 7️⃣ Optimización

In [ ]:
# Búsqueda en malla de políticas (q_lunes, q_miercoles)
grid = []
for ql in range(20, 61):
    for qw in range(0, 41):
        reps = replicate_policy(n_rep=250, weeks=100, q_lunes=ql, q_miercoles=qw, base_seed=5000)
        ciser = ci95(reps["servicio"])
        grid.append({
            "q_lunes": ql,
            "q_miercoles": qw,
            "servicio_prom": reps["servicio"].mean(),
            "costo_total_prom": reps["costo_total"].mean(),
            "ci_servicio_low": ciser["ci_low"],
            "ci_servicio_high": ciser["ci_high"],
        })

opt_df = pd.DataFrame(grid)
feasible = opt_df[opt_df["servicio_prom"] >= 0.80].copy()
best = feasible.sort_values("costo_total_prom").head(10)

best

In [ ]:
# Visualización tipo paper: frontera costo-servicio
plt.figure(figsize=(10, 6))
plt.scatter(opt_df["servicio_prom"], opt_df["costo_total_prom"], alpha=0.25, s=18, label="Políticas evaluadas")
plt.scatter(feasible["servicio_prom"], feasible["costo_total_prom"], alpha=0.35, s=22, label="Factibles (servicio ≥ 0.80)")
plt.axvline(0.80, color="red", linestyle="--", linewidth=2, label="Meta de servicio 80%")

best1 = best.iloc[0]
plt.scatter([best1["servicio_prom"]], [best1["costo_total_prom"]], color="black", s=120, marker="*", label="Mejor política")
plt.title("Frontera costo-servicio para políticas de reabastecimiento")
plt.xlabel("Nivel de servicio esperado")
plt.ylabel("Costo total esperado")
plt.legend()
plt.tight_layout()
plt.show()

print("Mejor política encontrada:")
print(best1)

## 8️⃣ Generación automática del PDF

In [ ]:
# Genera un PDF resumen estilo artículo (si reportlab está disponible)
# !pip -q install reportlab

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm

output_pdf = "reporte_tarea1_estilo_articulo.pdf"
c = canvas.Canvas(output_pdf, pagesize=letter)
width, height = letter

c.setFont("Helvetica-Bold", 14)
c.drawString(2*cm, height - 2*cm, "Simulación avanzada de inventarios de leche")

c.setFont("Helvetica", 10)
text = c.beginText(2*cm, height - 3*cm)
text.textLine("Autor: Edward Calderon | Curso: Simulación Avanzada")
text.textLine("Resumen:")
text.textLine("Se evaluaron políticas de abastecimiento con simulación Monte Carlo,")
text.textLine("estimando nivel de servicio y costo total bajo incertidumbre de demanda.")
text.textLine("")
text.textLine("Resultados clave (escenario base y mejor política factible):")
text.textLine(f"- Base (30,0): servicio medio={base_reps['servicio'].mean():.3f}, costo medio=${base_reps['costo_total'].mean():,.0f}")
text.textLine(f"- Mejor política: lunes={int(best1['q_lunes'])}, miércoles={int(best1['q_miercoles'])}, ")
text.textLine(f"  servicio={best1['servicio_prom']:.3f}, costo=${best1['costo_total_prom']:,.0f}")

c.drawText(text)
c.showPage()
c.save()

print(f"PDF generado: {output_pdf}")

---

### Conclusión ejecutiva

1. El contrato fijo de 30 cajas solo los lunes se puede evaluar con IC95% para verificar si cumple la meta de servicio del 80%.
2. Incluir entrega los miércoles permite reducir faltantes y, en muchos escenarios, bajar costo total al disminuir inventario ocioso acumulado.
3. La decisión final debe tomarse con base en la política factible de menor costo y su intervalo de confianza de servicio.